# Experimento final: proteção de corredores de migração (sem pesos cooperativos)

Versão final para o artigo. **Os pesos α/β/γ foram retirados**: todo ISP roteia pelos $k$ caminhos mais curtos por distância (First-Fit disaster-aware simples). A cooperação só age pela **reserva de espectro** nos enlaces do corredor de migração que o grupo cooperante prova serem exclusivos (σ(e) divulgado pelo mediador).

| Variante | Depende de $K$? | Rodadas por cenário base |
|---|---|---|
| sem reserva (First-Fit disaster-aware) | não | 1 |
| First-Fit (não disaster-aware) | não | 1 |
| reserva fixa $p$%, **cooperativa** | sim | 32 por $p$ |
| reserva fixa $p$%, **individual** | não | 1 por $p$ |
| reserva pela demanda, **cooperativa** | sim | 32 |
| reserva pela demanda, **individual** | não | 1 |

Sem pesos, a simulação de uma variante que não depende de $K$ é idêntica para todo $K$ (é determinística). Essas rodam só com $K=\emptyset$ e os resultados são expandidos para os 32 conjuntos com `expandir_para_todos_k`. Uma rodada extra (individual 40%, $K=\mathcal{I}$) confere essa equivalência.

Resultados em `output/cooperation_experiment_v3*`. Com as 6 porcentagens, são ~233 rodadas por cenário base.

In [ ]:
import json
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from experiment_worker import metrics_path, run_cooperation_experiment
from run_cooperation_experiment import (
    COOP_PATH,
    PROTECOES,
    consolidar,
    montar_tarefas,
    nome_reserva,
    pasta_saida,
)
from simulador.analysis.cooperation_metrics import expandir_para_todos_k

%matplotlib inline

## Parâmetros

In [ ]:
BASE_INDICES = [0]                       # list(range(10)) for all base scenarios
PORCENTAGENS = [10, 25, 40, 55, 70, 85]
INCLUIR_DEMANDA = True
MAX_WORKERS = min(os.cpu_count() or 4, 8)

REFERENCIA = pasta_saida(None, None, sem_pesos=True)   # no reservation (+ First-Fit)

# (mecanismo, escopo, nome em PROTECOES)
MECANISMOS = [(f"{p}%", nome_reserva(p), nome_reserva(p, isolado=True)) for p in PORCENTAGENS]
if INCLUIR_DEMANDA:
    MECANISMOS.append(("demanda", "demanda", "demanda_isolado"))

def tarefas_para(pasta, protecao, somente_k_vazio, first_fit=False, rotulo=""):
    pasta.mkdir(parents=True, exist_ok=True)
    out = montar_tarefas(BASE_INDICES, pasta, first_fit=first_fit, somente_k_vazio=somente_k_vazio)
    for t in out:
        t.update(output_dir=str(pasta), protecao=protecao, sem_pesos=True, rotulo=rotulo)
    return out

tarefas = tarefas_para(REFERENCIA, None, True, first_fit=True, rotulo="sem reserva")
for mec, coop, ind in MECANISMOS:
    tarefas += tarefas_para(pasta_saida(None, coop, True), PROTECOES[coop], False, rotulo=f"{mec} cooperativa")
    tarefas += tarefas_para(pasta_saida(None, ind, True), PROTECOES[ind], True, rotulo=f"{mec} individual")

# Check that K-independent variants really don't depend on K: one extra run
VERIFICACAO = nome_reserva(40, isolado=True)
for b in BASE_INDICES[:1]:
    pasta = pasta_saida(None, VERIFICACAO, True)
    todos = tuple(range(5))
    if not metrics_path(pasta, todos, b).exists():
        tarefas.append({"base_path": COOP_PATH.format(i=b), "base_index": b, "cooperantes": todos,
                        "variante": "cooperacao", "output_dir": str(pasta),
                        "protecao": PROTECOES[VERIFICACAO], "sem_pesos": True, "rotulo": "verificação"})

for idx, t in enumerate(tarefas):
    t.update(run_index=idx, total_runs=len(tarefas))
print(f"{len(tarefas)} rodadas pendentes | bases {BASE_INDICES} | {MAX_WORKERS} processos")

## Rodar

Rodadas prontas são puladas; pode interromper e continuar.

In [ ]:
inicio = datetime.now()
falhas = []
if tarefas:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(run_cooperation_experiment, t): t for t in tarefas}
        for n, future in enumerate(as_completed(futures), start=1):
            r = future.result()
            estado = "✓" if r["success"] else "✗"
            print(f"[{n}/{len(tarefas)}] {estado} {futures[future]['rotulo']} base={r['base_index']} {r['K']}  ({datetime.now() - inicio})")
            if not r["success"]:
                falhas.append(r)
print(f"Duração: {datetime.now() - inicio} | falhas: {len(falhas)}")

## Conferir: variantes individuais não dependem de $K$

In [ ]:
b = BASE_INDICES[0]
pasta = pasta_saida(None, VERIFICACAO, True)
a = json.loads(metrics_path(pasta, (), b).read_text())["summary"]
z = json.loads(metrics_path(pasta, tuple(range(5)), b).read_text())["summary"]
campos = ["eta_todos", "B_n", "B_m", "B_n_janela"]
print({c: (a[c], z[c]) for c in campos})
assert all(abs(a[c] - z[c]) < 1e-12 for c in campos), "individual protection changed with K!"
print("✓ idêntico para K = ∅ e K = I")

## Carregar

In [ ]:
def ler_json(pasta, padrao):
    return [json.loads(f.read_text()) for f in sorted(pasta.glob(padrao))]

def isp_ids_da_base(b):
    import pickle
    with open(COOP_PATH.format(i=b), "rb") as f:
        return sorted(isp.isp_id for isp in pickle.load(f).lista_de_isps)

IDS = {b: isp_ids_da_base(b) for b in BASE_INDICES}
resumos, linhas_isp = [], []

def adicionar_expandido(resultados, mecanismo, escopo):
    for r in resultados:
        if r["base_index"] not in IDS or r["K"] not in ("K_none", "first_fit"):
            continue
        s, p = expandir_para_todos_k(r, IDS[r["base_index"]])
        resumos.extend({**x, "mecanismo": mecanismo, "escopo": escopo} for x in s)
        linhas_isp.extend({**x, "mecanismo": mecanismo, "escopo": escopo} for x in p)

def adicionar_direto(resultados, mecanismo, escopo):
    for r in resultados:
        if r["base_index"] not in IDS:
            continue
        resumos.append({**r["summary"], "mecanismo": mecanismo, "escopo": escopo})
        linhas_isp.extend({**x, "mecanismo": mecanismo, "escopo": escopo} for x in r["per_isp"])

adicionar_expandido(ler_json(REFERENCIA, "K_none/metrics_*.json"), "sem reserva", "—")
adicionar_expandido(ler_json(REFERENCIA, "first_fit/metrics_*.json"), "First-Fit", "—")
for mec, coop, ind in MECANISMOS:
    adicionar_direto(ler_json(pasta_saida(None, coop, True), "K_*/metrics_*.json"), mec, "cooperativa")
    adicionar_expandido(ler_json(pasta_saida(None, ind, True), "K_none/metrics_*.json"), mec, "individual")

resumo = pd.DataFrame(resumos)
por_isp = pd.DataFrame(linhas_isp)
I = len(next(iter(IDS.values())))
ORDEM_MEC = ["sem reserva", "First-Fit"] + [m for m, _, _ in MECANISMOS]
resumo.to_csv(REFERENCIA / "resumo_final.csv", index=False)
por_isp.to_csv(REFERENCIA / "resumo_final_por_isp.csv", index=False)
resumo.groupby(["mecanismo", "escopo"]).size()

## Ganho e custo

Diferenças em relação a **sem reserva** (First-Fit disaster-aware), mesmo $K$ e mesma base. Média sobre os 32 conjuntos e o caso com todos cooperando.

In [ ]:
ref = resumo[resumo["mecanismo"] == "sem reserva"].set_index(["base_index", "K"])
linhas = []
for (mec, esc), g in resumo.groupby(["mecanismo", "escopo"]):
    g = g.set_index(["base_index", "K"])
    d_eta = g["eta_todos"] - ref["eta_todos"]
    d_bn = g["B_n"] - ref["B_n"]
    todos = g[g["k"] == I]
    linhas.append({
        "mecanismo": mec, "escopo": esc,
        "eta médio": g["eta_todos"].mean(), "d_eta médio": d_eta.mean(), "d_B_n médio": d_bn.mean(),
        "eta K=I": todos["eta_todos"].mean(), "B_n K=I": todos["B_n"].mean(), "B_m K=I": todos["B_m"].mean(),
        "slots reservados (média)": pd.to_numeric(g.get("prot_reserva_media_slots"), errors="coerce").mean(),
    })
tabela = pd.DataFrame(linhas)
tabela["mecanismo"] = pd.Categorical(tabela["mecanismo"], ORDEM_MEC, ordered=True)
tabela.sort_values(["mecanismo", "escopo"]).set_index(["mecanismo", "escopo"]).round(4)

## Por tamanho de $K$ (η de todos os ISPs)

In [ ]:
t = resumo.groupby(["mecanismo", "escopo", "k"])["eta_todos"].mean().unstack("k")
t.reindex(ORDEM_MEC, level=0).round(4)

## Gráficos

Três figuras: todos os ISPs, membros de $K$ e fora de $K$. Nas duas últimas, o bloqueio de fundo é mostrado como diferença para a referência no mesmo $K$ (o grupo muda de composição com $|K|$).

In [ ]:
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MECS = [m for m, _, _ in MECANISMOS]
CORES = {m: PALETA[i % len(PALETA)] for i, m in enumerate(MECS)}
TEXTO, GRADE, CINZA = "#52514e", "#e5e4df", "#52514e"
ESCOPOS = [("cooperativa", "-"), ("individual", "--")]

def estilo(ax, titulo, ylabel):
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_xlabel("|K| (provedores cooperando)", color=TEXTO)
    ax.set_ylabel(ylabel, color=TEXTO)
    ax.set_xticks(range(I + 1))
    ax.tick_params(colors=TEXTO)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

def serie(mec, esc, col, delta=False):
    g = resumo[(resumo["mecanismo"] == mec) & (resumo["escopo"] == esc)].set_index(["base_index", "K"])
    v = g[col] - ref[col] if delta else g[col]
    return v.groupby(g["k"]).mean().dropna()

def figura(col_eta, col_bn, grupo, arquivo, delta_bn):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    ax = axes[0]
    m = serie("sem reserva", "—", col_eta)
    ax.plot(m.index, m.values, color=CINZA, linewidth=2, marker="o", markersize=8, label="sem reserva (First-Fit disaster-aware)")
    for mec in MECS:
        for esc, traco in ESCOPOS:
            m = serie(mec, esc, col_eta)
            ax.plot(m.index, m.values, traco, color=CORES[mec], linewidth=2,
                    marker="o" if esc == "cooperativa" else None, markersize=8, label=f"{mec} {esc}")
    estilo(ax, f"Fração migrada ({grupo})", "η médio")

    ax = axes[1]
    if delta_bn:
        ax.axhline(0, color=TEXTO, linewidth=1)
    else:
        m = serie("sem reserva", "—", col_bn)
        ax.plot(m.index, m.values, color=CINZA, linewidth=2, marker="o", markersize=8)
    for mec in MECS:
        for esc, traco in ESCOPOS:
            m = serie(mec, esc, col_bn, delta=delta_bn)
            ax.plot(m.index, m.values, traco, color=CORES[mec], linewidth=2,
                    marker="o" if esc == "cooperativa" else None, markersize=8)
    estilo(ax, f"{'Δ ' if delta_bn else ''}Bloqueio de fundo ({grupo})", "ΔB_n vs sem reserva" if delta_bn else "B_n médio")

    ax = axes[2]
    ax.axhline(0, color=TEXTO, linewidth=1)
    for mec in MECS:
        for esc, traco in ESCOPOS:
            m = serie(mec, esc, col_eta, delta=True)
            ax.plot(m.index, m.values, traco, color=CORES[mec], linewidth=2,
                    marker="o" if esc == "cooperativa" else None, markersize=8)
    estilo(ax, f"Ganho sobre sem reserva ({grupo})", "Δη (mesmo K)")

    h, l = axes[0].get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0.16, 1, 1))
    fig.legend(h, l, frameon=False, loc="lower center", ncol=min(len(l), 6), bbox_to_anchor=(0.5, 0.0))
    fig.savefig(REFERENCIA / arquivo, dpi=150, bbox_inches="tight")
    plt.show()

figura("eta_todos", "B_n", "todos os ISPs", "final_todos.png", delta_bn=False)
figura("eta_K", "B_n_K", "membros de K", "final_membros_K.png", delta_bn=True)
figura("eta_isolados", "B_n_isolados", "fora de K", "final_fora_K.png", delta_bn=True)

## Por ISP com todos cooperando

In [ ]:
x = por_isp[por_isp["k"] == I].pivot_table(index=["mecanismo", "escopo"], columns="isp_id", values="eta_i")
x.reindex(ORDEM_MEC, level=0).round(3)